# Huynen decomposition

This tutorial applies the Huynen decomposition to the San Francisco ALOS-1 image and saves the result. The function accepts Sinclair (`S`), covariance (`C3`), or coherency (`T3`) datasets and returns a matrix dataset.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from dask.diagnostics import ProgressBar

from polsarpro.decompositions import huynen
from polsarpro.io import open_netcdf_beam, polmat_to_netcdf
from polsarpro.util import S_to_T3, multilook

# Change to your data paths.
input_file = Path("/data/psp/test_files/SAN_FRANCISCO_ALOS1_slc.nc")
output_dir = Path("/data/psp/res")

## Load data

Open the SNAP NetCDF-BEAM scattering matrix. See the Quickstart tutorial for preparing an input dataset.

In [ ]:
S = open_netcdf_beam(input_file)
S

## Prepare the coherency matrix

Convert to T3 and apply four azimuth looks and one range look to reduce the number of image lines.

In [ ]:
T3 = S_to_T3(S)
T3 = multilook(T3, dim_az=4, dim_rg=1)
T3

## Apply the decomposition and save the target

Use a 5 × 5 spatial averaging window. `output_basis="pauli"` (the default) returns a T3 dataset; `output_basis="lexicographic"` returns C3. Either output basis is available for every supported input type.

Save the matrix with `polmat_to_netcdf`. An existing output file is overwritten.

In [ ]:
target = huynen(T3, boxcar_size=(5, 5), output_basis="pauli")
file_out = output_dir / "psp_huynen.nc"
output_dir.mkdir(parents=True, exist_ok=True)

with ProgressBar():
    polmat_to_netcdf(target, file_out)

## Display the target matrix

Reopen the saved dataset with `open_netcdf_beam`.

In [ ]:
result = open_netcdf_beam(file_out)
result

Display the diagonal elements. Adjust the color limits for your dataset.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(12, 7), constrained_layout=True)
for ax, name in zip(axes, ("m11", "m22", "m33")):
    result[name].plot.imshow(ax=ax, vmin=0, vmax=1)
    ax.set_title(f"Target {name}")
plt.show()